In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv1D, MaxPooling1D

# Load the training and test datasets
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')


In [3]:
# Feature Engineering
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1

In [4]:
# Preprocessing
train_df = train_df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
test_df = test_df.drop(['Name', 'Ticket', 'Cabin'], axis=1)

In [5]:
# Impute missing values
train_df['Age'].fillna(train_df['Age'].mean(), inplace=True)
test_df['Age'].fillna(test_df['Age'].mean(), inplace=True)
test_df['Fare'].fillna(test_df['Fare'].mean(), inplace=True)

In [6]:
# One-hot encoding
train_df = pd.get_dummies(train_df, columns=['Pclass', 'Sex', 'Embarked'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['Pclass', 'Sex', 'Embarked'], drop_first=True)

In [7]:
# Split into training and validation sets
X = train_df.drop(['Survived'], axis=1)
y = train_df['Survived']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

In [8]:
# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [9]:
# Reshape the data to fit the requirements of a 1D Convolutional Neural Network
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_val = np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1))

In [10]:
# Build the model
model = Sequential()
model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

In [11]:
# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])


In [12]:
# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

Epoch 1/50
23/23 [==============================] - 1s 14ms/step - loss: 0.6179 - accuracy: 0.6854 - val_loss: 0.5138 - val_accuracy: 0.8156
Epoch 2/50
23/23 [==============================] - 0s 4ms/step - loss: 0.5254 - accuracy: 0.7683 - val_loss: 0.4395 - val_accuracy: 0.8268
Epoch 3/50
23/23 [==============================] - 0s 4ms/step - loss: 0.4805 - accuracy: 0.7767 - val_loss: 0.4102 - val_accuracy: 0.8212
Epoch 4/50
23/23 [==============================] - 0s 4ms/step - loss: 0.4641 - accuracy: 0.7823 - val_loss: 0.4026 - val_accuracy: 0.8212
Epoch 5/50
23/23 [==============================] - 0s 4ms/step - loss: 0.4516 - accuracy: 0.7963 - val_loss: 0.3961 - val_accuracy: 0.8380
Epoch 6/50
23/23 [==============================] - 0s 4ms/step - loss: 0.4511 - accuracy: 0.7978 - val_loss: 0.3929 - val_accuracy: 0.8268
Epoch 7/50
23/23 [==============================] - 0s 4ms/step - loss: 0.4380 - accuracy: 0.8230 - val_loss: 0.3907 - val_accuracy: 0.8268
Epoch 8/50
23/23 [=

In [13]:
# Evaluate the model
scores = model.evaluate(X_val, y_val)
print("\n%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

6/6 [==============================] - 0s 13ms/step - loss: 0.4054 - accuracy: 0.8324

accuracy: 83.24%


In [14]:
# Make predictions on the test set
X_test = test_df.drop(['PassengerId'], axis=1)
X_test = scaler.transform(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
y_pred = model.predict(X_test)
y_pred = [int(round(x[0])) for x in y_pred]

In [15]:
# Prepare the submission file
submission = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Survived': y_pred})
submission.to_csv('submission.csv', index=False)

🚀 Was it helpful? thanks you for upvote :)

👉 [Check other of my models ](https://www.kaggle.com/roxket/code)